# PaperMind — Notebook 2: Router Query Engine

Notebook 1 indexed a single paper. In real PaperMind use, a user will have a library of papers — and asking a question against *all of them blended together* is wasteful and noisy. A `RouterQueryEngine` solves this: each paper gets its own index, and an LLM-driven selector picks which paper to query based on the question.

**What we build here:**
1. Re-use the env/LLM/embeddings setup from notebook 1
2. Download two papers — *Attention Is All You Need* and *BERT*
3. Build a separate `VectorStoreIndex` per paper, persisted to its own `../indexes/<name>/`
4. Wrap each index's query engine in a `QueryEngineTool` with a description
5. Combine them in a `RouterQueryEngine` with `LLMSingleSelector` — the LLM reads each tool's description and the user's query, then picks the best tool
6. Run 5 test queries (2 clearly per paper + 1 ambiguous) and inspect the selector's choice and stated reason

**Why routing matters:** routing keeps each retrieval focused, makes citations unambiguous (you know which paper an answer came from), and scales — adding a new paper means adding one tool, not re-embedding everything.

## 1. Setup — env, async patch, LLM, embeddings

We're using **Groq** here instead of Gemini. Groq's free tier has higher RPM limits (~30 vs Gemini Flash's 5), which matters for router engines because each query fires multiple LLM calls (selector + synthesis).

We use `llama-3.3-70b-versatile` — a strong open model hosted on Groq's fast inference infrastructure. The HuggingFace BGE embeddings stay the same; only the LLM changes.

`nest_asyncio.apply()` is still needed because LlamaIndex calls `asyncio.run()` internally and Jupyter already has a running event loop.

In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv
import nest_asyncio

nest_asyncio.apply()

load_dotenv("../.env")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
assert GROQ_API_KEY, "GROQ_API_KEY not found in ../.env — get a free key at console.groq.com"

from llama_index.llms.groq import Groq
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import Settings

llm = Groq(model="llama-3.3-70b-versatile", api_key=GROQ_API_KEY)
embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-base-en-v1.5")
Settings.llm = llm
Settings.embed_model = embed_model

print("LLM (Groq) + embeddings configured")

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

LLM (Groq) + embeddings configured


## 2. Download both papers

- *Attention Is All You Need* (Vaswani et al., 2017) — introduces the Transformer
- *BERT* (Devlin et al., 2018) — introduces bidirectional pre-training with masked language modeling

Both papers cover attention and transformer-style architectures, which is what makes routing interesting: they overlap in some areas, so the selector has to reason about *what specifically* the user is asking.

In [2]:
import requests

DATA_DIR = Path("../data")
INDEX_ROOT = Path("../indexes")
DATA_DIR.mkdir(parents=True, exist_ok=True)
INDEX_ROOT.mkdir(parents=True, exist_ok=True)

papers = {
    "attention": {
        "url": "https://arxiv.org/pdf/1706.03762",
        "file": DATA_DIR / "attention.pdf",
        "index_dir": INDEX_ROOT / "attention",
    },
    "bert": {
        "url": "https://arxiv.org/pdf/1810.04805",
        "file": DATA_DIR / "bert.pdf",
        "index_dir": INDEX_ROOT / "bert",
    },
}

for name, p in papers.items():
    if p["file"].exists():
        print(f"[{name}] already on disk: {p['file'].stat().st_size / 1024:.1f} KB")
    else:
        r = requests.get(p["url"], timeout=30)
        r.raise_for_status()
        p["file"].write_bytes(r.content)
        print(f"[{name}] downloaded: {p['file'].stat().st_size / 1024:.1f} KB")

[attention] already on disk: 2163.3 KB
[bert] already on disk: 757.0 KB


## 3. Helper — load a PDF and build/reload its index

This is the reusable pattern from notebook 1, factored into one function. Every later notebook can copy it.

- Use `PyMuPDF` (`fitz`) to extract clean text per page (the default `pypdf` mangles arXiv layouts).
- If the index dir already has files, reload from disk.
- Otherwise embed + persist.

In [3]:
import fitz  # pymupdf
from llama_index.core import (
    Document,
    VectorStoreIndex,
    StorageContext,
    load_index_from_storage,
)


def load_pdf_as_documents(pdf_path: Path) -> list[Document]:
    pdf = fitz.open(str(pdf_path))
    docs = [
        Document(
            text=page.get_text(),
            metadata={"page": i + 1, "source": pdf_path.name},
        )
        for i, page in enumerate(pdf)
    ]
    pdf.close()
    return docs


def build_or_load_index(pdf_path: Path, index_dir: Path) -> VectorStoreIndex:
    index_dir.mkdir(parents=True, exist_ok=True)
    if any(index_dir.iterdir()):
        storage_context = StorageContext.from_defaults(persist_dir=str(index_dir))
        index = load_index_from_storage(storage_context)
        print(f"  loaded existing index from {index_dir}")
    else:
        docs = load_pdf_as_documents(pdf_path)
        index = VectorStoreIndex.from_documents(docs)
        index.storage_context.persist(persist_dir=str(index_dir))
        print(f"  built new index ({len(docs)} pages) and persisted to {index_dir}")
    return index

## 4. Build a separate index + query engine per paper

Each paper is its own self-contained vector store — independent embeddings, independent retrieval. `similarity_top_k=3` is the same default we used before.

In [4]:
engines = {}
for name, p in papers.items():
    print(f"[{name}]")
    index = build_or_load_index(p["file"], p["index_dir"])
    engines[name] = index.as_query_engine(similarity_top_k=3)

print("\nQuery engines ready:", list(engines.keys()))

[attention]
  loaded existing index from ../indexes/attention
[bert]
  loaded existing index from ../indexes/bert

Query engines ready: ['attention', 'bert']


## 5. Wrap each engine in a QueryEngineTool

A `QueryEngineTool` is just `(query_engine, name, description)`. **The `description` is what the LLM router reads** to decide where to send a query — so write it like a docstring aimed at a model:

- Be specific about *what topics* this paper covers.
- Include distinguishing keywords (model names, techniques, authors, years).
- Avoid generic phrases like "useful for NLP questions" — they don't help the router disambiguate.

Compare the two descriptions below: the topic overlap (attention, transformers) is real, but each description names what makes its paper *distinct*.

In [5]:
from llama_index.core.tools import QueryEngineTool

attention_tool = QueryEngineTool.from_defaults(
    query_engine=engines["attention"],
    name="attention_paper",
    description=(
        "Useful for questions about the original Transformer architecture introduced in "
        "'Attention Is All You Need' (Vaswani et al., 2017). Topics include: scaled "
        "dot-product attention, multi-head self-attention, encoder-decoder stacks, "
        "sinusoidal positional encoding, and machine-translation experiments on "
        "WMT 2014 EN-DE / EN-FR."
    ),
)

bert_tool = QueryEngineTool.from_defaults(
    query_engine=engines["bert"],
    name="bert_paper",
    description=(
        "Useful for questions about BERT (Devlin et al., 2018) — a bidirectional "
        "transformer encoder pre-trained with masked language modeling (MLM) and "
        "next-sentence prediction (NSP), then fine-tuned on downstream NLP tasks. "
        "Topics include: pre-training objectives, WordPiece tokenization, "
        "fine-tuning on GLUE / SQuAD, and BERT-Base vs BERT-Large."
    ),
)

tools = [attention_tool, bert_tool]

## 6. Build the RouterQueryEngine

**`LLMSingleSelector`** asks the LLM to pick exactly one tool per query. It's the right choice when each query is about *one* paper — using `LLMMultiSelector` would let it pick several, which is useful for cross-paper synthesis but adds latency and cost.

Under the hood the selector sends the LLM a prompt like:

> Given the choices below... which choice is most relevant? Return the index and reason.

...with each tool's description as a numbered choice. The LLM responds with a structured selection containing an index and a free-text reason.

In [6]:
from llama_index.core.query_engine import RouterQueryEngine
from llama_index.core.selectors import LLMSingleSelector

router_engine = RouterQueryEngine(
    selector=LLMSingleSelector.from_defaults(llm=llm),
    query_engine_tools=tools,
)

tool_names = [t.metadata.name for t in tools]
print("Router built. Available tools:", tool_names)

Router built. Available tools: ['attention_paper', 'bert_paper']


## 7. Test queries — inspect routing + reasoning

Five queries:
- 2 that should route to *Attention Is All You Need*
- 2 that should route to *BERT*
- 1 ambiguous — multi-head self-attention is introduced in the Transformer paper but BERT uses it too, so the router has to decide based on the description

After each query, we inspect `response.metadata['selector_result']` to print which tool the LLM picked and the reason it gave.

In [7]:
import time

queries = [
    # → attention_paper
    "How are sinusoidal positional encodings computed in the original Transformer?",
    "What were the BLEU scores reported on WMT 2014 English-to-German translation?",
    # → bert_paper
    "What is the masked language modeling objective and what fraction of tokens is masked?",
    "How does BERT-Large perform on the GLUE benchmark compared to prior work?",
    # ambiguous — both papers discuss multi-head attention
    "How does multi-head self-attention work?",
]

# Groq's free tier is generous (~30 RPM), but a small sleep is still cheap
# insurance against bursts when refine-synthesis fires several calls per query.
SLEEP_BETWEEN_QUERIES = 2  # seconds

for i, q in enumerate(queries):
    print("=" * 100)
    print(f"Q: {q}\n")

    response = router_engine.query(q)

    selector_result = response.metadata.get("selector_result")
    if selector_result is not None:
        for sel in selector_result.selections:
            print(f"→ Routed to: {tool_names[sel.index]}")
            print(f"→ Reason:    {sel.reason}")
    else:
        print("(no selector metadata available on response)")

    print(f"\nA: {response}\n")

    if i < len(queries) - 1:
        time.sleep(SLEEP_BETWEEN_QUERIES)

Q: How are sinusoidal positional encodings computed in the original Transformer?

→ Routed to: attention_paper
→ Reason:    The question is about the original Transformer architecture, specifically sinusoidal positional encoding, which is mentioned in choice 1.

A: Sinusoidal positional encodings are computed using sine and cosine functions of different frequencies. The computation is as follows: 
PE(pos,2i) = sin(pos/10000^(2i/dmodel)) 
PE(pos,2i+1) = cos(pos/10000^(2i/dmodel)) 
where pos is the position and i is the dimension. This allows the model to easily learn to attend by relative positions.

Q: What were the BLEU scores reported on WMT 2014 English-to-German translation?

→ Routed to: attention_paper
→ Reason:    The question is related to machine-translation experiments on WMT 2014, which is mentioned in choice 1 as a topic, specifically the English-to-German translation.

A: The BLEU scores reported on WMT 2014 English-to-German translation were 27.3 for the Transformer base 

## When is routing the right tool?

**Use a router when:**
- Each query is naturally about one document (or one *kind* of document) and you want clean citations.
- Your document set is heterogeneous — papers, code, meeting notes, contracts. Blending their embeddings into one index makes retrieval worse, not better.
- You want to scale to many sources without re-embedding everything when one is added.

**Don't use a router when:**
- Queries genuinely need to synthesize across documents ("compare paper A and paper B"). For that, use `LLMMultiSelector` or build a higher-level summarizing engine on top.
- Your documents are small and homogeneous — one shared index is simpler and the routing latency / extra LLM call isn't worth it.

**Tuning the router:** the single biggest lever is the **tool descriptions**. If you see misroutes, rewrite the descriptions to be more specific, name distinguishing concepts, and avoid overlap. The retrieval quality of each underlying engine is the second lever; the LLM selector itself rarely needs tuning.